# `compound_lemma_processor` — Usage Notebook

Demonstrates the core algorithm behind `scripts/compound_lemma_processor.py`.

**What the script does**

Compound nouns in the corpus appear as groups of `N` / `N;@2` / `N;@3` … sibling
lines that share the same syntactic prefix up to a bare `N` marker column. The
script inserts a shared lemma ID at that marker column and creates/refines the
corresponding dictionary entry.

**Algorithm**
1. *(Optional)* **NP expansion**: detect NPs whose direct children are all `N`-at
   tags, insert a grouping bare `N` before them.
2. **Group detection** (`find_compound_groups`): find all consecutive sequences
   of lines with a shared marker-`N` column and component lemma IDs.
3. **Layered pairing** (`pair_components_layered`): for k ≥ 2 components, pair
   left-to-right iteratively: (A,B)→AB, (AB,C)→ABC, …
4. **ID insertion**: insert the outermost compound ID at `marker_col+1`.

All cells read from `data/` — no files are modified.

---
## 0. Setup

In [ ]:
import sys
import os
import re
from collections import defaultdict

sys.path.insert(0, os.path.join('..', 'src'))

DICT_FILE   = os.path.join('..', 'data', 'dict', 'dictionary.txt')
CORPUS_FILE = os.path.join('..', 'data', 'trees', 'MYS_01.txt')
TEXT_DIR    = os.path.join('..', 'data', 'text')

print("Python:", sys.version.split()[0])

Python: 3.13.3


---
## 1. Corpus Line Structure for Compound Nouns

In the corpus a two-noun compound looks like:

```
IP-MAT,NP,N,L050001,N,L050002,LOG,ama
IP-MAT,NP,N,L050001,N;@2,L050003,LOG,tu
```

- The bare `N` at position 2 is the **marker column** — this is where the compound
  ID will be inserted.
- `N` at position 4 (component 1) and `N;@2` at position 4 (component 2) each carry
  their own lemma ID in the slot immediately after them.
- After processing, a compound ID `L050NNN` is inserted at position 3 (marker_col + 1):

```
IP-MAT,NP,N,L05NNN,L050001,N,L050002,LOG,ama
IP-MAT,NP,N,L05NNN,L050001,N;@2,L050003,LOG,tu
```

In [2]:
from oncoj.corpus import CorpusDocument
from oncoj.tags import strip_disambig

LEMMA_PAT = re.compile(r'^[A-Za-z]\d+[a-z]*$')

def _fields(line: str) -> list:
    return line.rstrip('\n').split(',')

def _is_n_at(tag: str) -> bool:
    return strip_disambig(tag) == 'N'

# Show raw lines that already have a compound structure
doc = CorpusDocument.from_file(CORPUS_FILE)

# Find lines where a bare N is followed immediately by another N or N;@k
compound_candidates = []
raw_lines = open(CORPUS_FILE, encoding='utf-8').readlines()

for i, raw in enumerate(raw_lines):
    f = _fields(raw)
    # Look for C-N or C-NP tags — these indicate annotated compounds
    if any(strip_disambig(t) in ('C-N', 'C-NP') for t in f):
        compound_candidates.append((i+1, raw.rstrip()))

print(f"Lines with C-N or C-NP tags in EN_01.txt: {len(compound_candidates)}")
print("\nFirst 5 examples:")
for lno, line in compound_candidates[:5]:
    print(f"  line {lno:4d}: {line[:90]}")

Lines with C-N or C-NP tags in EN_01.txt: 179

First 5 examples:
  line    6: IP-MAT,IP-ARG,C-NP,N,N,LOG,kamu
  line    7: IP-MAT,IP-ARG,C-NP,N,N;@2,LOG,nusi
  line    8: IP-MAT,IP-ARG,C-NP;@5,N,N,LOG,papuri
  line    9: IP-MAT,IP-ARG,C-NP;@5,N,N;@2,LOG,ra
  line   28: IP-MAT,IP-NMZ-FRM,PP-DAT,NP,PP-GEN,NP,IP-REL,IP-ADV,NP,N,C-N,PP-GEN,NP,N,LOG,sumyemutukamu


---
## 2. Finding the Marker Column

The `_find_marker_col()` function locates the column index of the bare `N` that acts
as the compound group marker.  The contract:

- The marker `N` must NOT be immediately followed by a lemma ID (otherwise the compound
  is already annotated).
- The next field after the marker `N` must be another `N`-at tag (a component).
- That component N-at tag must be immediately followed by a lemma ID.

In [ ]:
def _find_marker_col(fields: list) -> int | None:
    """
    Return the index of the bare N that acts as compound marker, or None.
    Mirrors the logic from compound_lemma_processor._find_marker_col().
    """
    for col in range(len(fields) - 3):
        tag = fields[col]
        if tag != 'N':
            continue
        # col+1 must not be a lemma ID (otherwise already annotated)
        if LEMMA_PAT.match(fields[col + 1]):
            continue
        # col+1 must be an N-at component tag
        if not _is_n_at(fields[col + 1]):
            continue
        # col+2 must be a lemma ID (the component's existing annotation)
        if not LEMMA_PAT.match(fields[col + 2]):
            continue
        return col
    return None

# Test on a few lines
test_lines = [
    "IP-MAT,NP,N,L050001,N,L050002,LOG,ama",         # has marker at col 2 → but already has ID!
    "IP-MAT,NP,N,N,L050002,LOG,ama",                  # marker at col 2, component at col 3
    "IP-MAT,NP,N,N;@2,L050003,LOG,tu",               # sibling of above
    "IP-MAT,NP,N,LOG,ama",                            # plain noun — no compound
]

for line in test_lines:
    f = _fields(line)
    col = _find_marker_col(f)
    print(f"  marker_col={col}  {line[:70]}")

---
## 3. Group Detection

Import and use the script's `find_compound_groups()` to scan a real corpus file.

We replicate the group-detection logic here to show what it finds.

In [ ]:
# Load lines from EN_01.txt
with open(CORPUS_FILE, encoding='utf-8') as f:
    corpus_lines = f.readlines()

# ── Simplified group detection (mirrors the script) ───────────────────────────

def _component_from_fields(fields, marker_col):
    """Extract component info: comp_id and text_form."""
    # component N-at is at marker_col+1 (or marker_col for siblings)
    # Actual component slot depends on position; here we check the pattern:
    # fields[marker_col+1] = N-at tag, fields[marker_col+2] = comp_lemma_id,
    # fields[-2] = phon_tag, fields[-1] = word_form
    if len(fields) < marker_col + 3:
        return None
    comp_id = fields[marker_col + 2]
    if not LEMMA_PAT.match(comp_id):
        return None
    return {'comp_id': comp_id, 'text_form': fields[-1].rstrip()}


groups = []
n = len(corpus_lines)
i = 0
while i < n:
    f = _fields(corpus_lines[i])
    marker_col = _find_marker_col(f)
    if marker_col is None or f[marker_col] != 'N':
        i += 1
        continue
    # Already has compound ID?
    if LEMMA_PAT.match(f[marker_col + 1]):
        i += 1
        continue

    comp0 = _component_from_fields(f, marker_col)
    if comp0 is None:
        i += 1
        continue

    group_lines = [i]
    components  = [{'line_idx': i, 'at_tag': f[marker_col], **comp0}]

    j = i + 1
    while j < n:
        fj = _fields(corpus_lines[j])
        if len(fj) <= marker_col:
            break
        if fj[:marker_col] != f[:marker_col]:
            break
        at_tag = fj[marker_col]
        if not _is_n_at(at_tag) or at_tag == 'N':
            break
        if LEMMA_PAT.match(fj[marker_col + 1]):
            break
        compj = _component_from_fields(fj, marker_col)
        if compj is None:
            break
        group_lines.append(j)
        components.append({'line_idx': j, 'at_tag': at_tag, **compj})
        j += 1

    if len(group_lines) >= 2:
        groups.append({'start': i, 'end': j-1, 'marker_col': marker_col,
                       'lines': group_lines, 'components': components})
    i = j if len(group_lines) >= 2 else i + 1

print(f"Compound groups found in EN_01.txt: {len(groups)}")
print("\nSample groups:")
for g in groups[:5]:
    forms = [c['text_form'] for c in g['components']]
    ids   = [c['comp_id']   for c in g['components']]
    print(f"  lines {g['start']+1}–{g['end']+1}  marker_col={g['marker_col']}  "
          f"components={list(zip(ids, forms))}")

---
## 4. Layered Binary Pairing

For a group of k components, `pair_components_layered()` builds intermediate compound
IDs left-to-right:

- **k=2**: `(A, B) → AB_id`
- **k=3**: `(A, B) → AB_id`, then `(AB_id, C) → ABC_id`
- **k=4**: `(A, B) → AB_id`, `(C, D) → CD_id`, then `(AB_id, CD_id) → ABCD_id`

The final ID in the chain is inserted at `marker_col+1` in each line of the group.

In [ ]:
def pair_components_layered_demo(components: list[dict]) -> list[tuple]:
    """Simulate layered pairing; return list of (combined_form, layer)."""
    slots = [(c['comp_id'], c['text_form']) for c in components]
    results = []
    layer = 0
    new_id_counter = [50001]

    while len(slots) > 1:
        new_slots = []
        k = 0
        while k < len(slots):
            if k + 1 < len(slots):
                id1, form1 = slots[k]
                id2, form2 = slots[k + 1]
                new_id = f"L{new_id_counter[0]:06d}"
                new_id_counter[0] += 1
                combined_form = form1 + form2
                new_slots.append((new_id, combined_form))
                results.append((new_id, combined_form, layer))
                k += 2
            else:
                new_slots.append(slots[k])
                k += 1
        slots = new_slots
        layer += 1

    return results

# Test cases
examples = [
    [{'comp_id': 'L000100', 'text_form': 'ama'},
     {'comp_id': 'L000200', 'text_form': 'tu'}],
    [{'comp_id': 'L000100', 'text_form': 'taka'},
     {'comp_id': 'L000200', 'text_form': 'ma'},
     {'comp_id': 'L000300', 'text_form': 'para'}],
    [{'comp_id': 'L000100', 'text_form': 'a'},
     {'comp_id': 'L000200', 'text_form': 'si'},
     {'comp_id': 'L000300', 'text_form': 'pi'},
     {'comp_id': 'L000400', 'text_form': 'ki'}],
]

for comps in examples:
    results = pair_components_layered_demo(comps)
    input_forms = [c['text_form'] for c in comps]
    print(f"  Input: {input_forms}")
    for cid, form, layer in results:
        print(f"    layer {layer}: {cid} = '{form}'")
    outermost = results[-1]
    print(f"    → insert {outermost[0]} at marker column")
    print()

---
## 5. NP Expansion (Optional)

When `NP_EXPANSION = True`, the script first looks for NPs whose **all** immediate
direct children are `N`-at tags without an existing marker `N` above them.

For example:
```
...,NP,N,L000100,LOG,ama      ← N is a direct child of NP
...,NP,N;@2,L000200,LOG,tu   ← N;@2 is sibling
```
After NP expansion, a grouping bare `N` is inserted at the NP child level:
```
...,NP,N,N,L000100,LOG,ama
...,NP,N,N;@2,L000200,LOG,tu
```
This makes the group detectable by `find_compound_groups()`.

In [ ]:
# Find NP lines where all siblings are N-at (candidates for NP expansion)
NP_LIKE = re.compile(r'^NP(-[A-Z]+)*$')

np_expansion_candidates = defaultdict(list)  # (prefix, np_col) → [(lineno, line)]

for i, raw in enumerate(corpus_lines):
    f = _fields(raw)
    # Look for NP tag directly followed by N or N;@k
    for col, tag in enumerate(f[:-1]):
        if not NP_LIKE.match(strip_disambig(tag)):
            continue
        child_tag = f[col + 1]
        if _is_n_at(child_tag):
            key = (tuple(f[:col+1]), col)
            np_expansion_candidates[key].append((i+1, raw.rstrip()))
            break

# Filter to groups with 2+ siblings (true compound candidates)
multi_sibling = {k: v for k, v in np_expansion_candidates.items() if len(v) >= 2}
print(f"NP groups with 2+ N-at siblings: {len(multi_sibling)}")

# Show first candidate
if multi_sibling:
    key, lines = next(iter(multi_sibling.items()))
    np_path = ','.join(key[0])
    print(f"\nExample NP path: {np_path}")
    print("Lines:")
    for lno, line in lines[:4]:
        print(f"  {lno:4d}: {line[:85]}")
    print("  → After NP expansion, a bare N would be inserted before each N-at tag.")

---
## 6. Dictionary Interaction

For each compound pair, the script calls `resolve_compound()` which:
1. Checks if an entry whose `.FORM` matches the concatenated form already exists
   (`DICT_SEARCH = True`).
2. If found and missing `.COMPOUND` back-references, adds them (`DICT_REFINE = True`).
3. If not found, creates a new entry (`DICT_ENTRY_CREATE = True`).

The `.COMPOUND ref_target=<ID>  <form>` field uses the **dictionary's canonical
`.FORM`** (not the corpus text form) to ensure stable references.

Below we show how to check whether a compound form already exists in the dictionary.

In [ ]:
from oncoj.dictionary import Dictionary

d = Dictionary.from_file(DICT_FILE)

def find_compound_in_dict(form: str, dictionary: Dictionary) -> str | None:
    """Return the entry ID if the combined form exists in the dictionary."""
    for entry in dictionary.values():
        if form in entry.get_all('.FORM'):
            return entry.eid
    return None

# Test with compound forms from detected groups
print("Compound form lookups:")
test_forms = []
for g in groups[:10]:
    combined = ''.join(c['text_form'] for c in g['components'])
    test_forms.append(combined)

for form in set(test_forms):
    eid = find_compound_in_dict(form, d)
    status = f"found: {eid}" if eid else "not in dictionary → would create new entry"
    print(f"  {form:20s}  {status}")

In [ ]:
from oncoj.kana import phonemic_to_kana

def build_compound_entry_preview(
    new_id: str,
    comp1_id: str, comp1_dict_form: str,
    comp2_id: str, comp2_dict_form: str,
    combined_text_form: str,
) -> str:
    """Show what a new compound dictionary entry would look like."""
    combined_kana = phonemic_to_kana(combined_text_form)
    lines = [
        f"=== {new_id}",
        f".GLOSS\t{combined_text_form}",
        ".MEANING\t[compound noun]",
        f".FORM\t{combined_text_form}",
        f".KANA\t{combined_kana}",
        ".POS\tnoun",
        f".COMPOUND\tref_target={comp1_id}  {comp1_dict_form}",
        f".COMPOUND\tref_target={comp2_id}  {comp2_dict_form}",
    ]
    return '\n'.join(lines)

# Use first detected group
if groups:
    g = groups[0]
    c0, c1 = g['components'][0], g['components'][1]
    # Look up canonical .FORM from dictionary
    e0 = d.get(c0['comp_id'])
    e1 = d.get(c1['comp_id'])
    dict_form0 = e0.get_first('.FORM') if e0 else c0['text_form']
    dict_form1 = e1.get_first('.FORM') if e1 else c1['text_form']
    combined   = c0['text_form'] + c1['text_form']

    preview = build_compound_entry_preview(
        'L050001', c0['comp_id'], dict_form0, c1['comp_id'], dict_form1, combined
    )
    print("Simulated new compound dictionary entry:")
    print(preview)

---
## 7. Inserting the Compound ID into Corpus Lines

Once the outermost compound ID is resolved, it is inserted at `marker_col + 1` in
every line of the group.

In [ ]:
def insert_compound_id_into_line(line: str, marker_col: int, compound_id: str) -> str:
    f = _fields(line)
    f.insert(marker_col + 1, compound_id)
    return ','.join(f)

if groups:
    g = groups[0]
    compound_id = 'L050001'  # simulated
    print("Before insertion:")
    for line_idx in g['lines']:
        print(f"  {corpus_lines[line_idx].rstrip()[:90]}")
    print("\nAfter insertion:")
    for line_idx in g['lines']:
        new_line = insert_compound_id_into_line(
            corpus_lines[line_idx], g['marker_col'], compound_id
        )
        print(f"  {new_line[:90]}")

---
## 8. Corpus-Wide Statistics

How many compound groups would be detected across all corpus files?

In [ ]:
def count_groups_in_file(fpath: str) -> int:
    with open(fpath, encoding='utf-8') as f:
        lines = f.readlines()
    n = len(lines)
    count = 0
    i = 0
    while i < n:
        flds = _fields(lines[i])
        col = _find_marker_col(flds)
        if col is None or flds[col] != 'N' or LEMMA_PAT.match(flds[col + 1] if len(flds) > col + 1 else ''):
            i += 1
            continue
        comp0 = _component_from_fields(flds, col)
        if comp0 is None:
            i += 1
            continue
        # Count siblings
        j = i + 1
        sibling_count = 1
        while j < n:
            fj = _fields(lines[j])
            if len(fj) <= col or fj[:col] != flds[:col]:
                break
            at = fj[col]
            if not _is_n_at(at) or at == 'N':
                break
            if LEMMA_PAT.match(fj[col + 1] if len(fj) > col + 1 else ''):
                break
            if _component_from_fields(fj, col) is None:
                break
            sibling_count += 1
            j += 1
        if sibling_count >= 2:
            count += 1
            i = j
        else:
            i += 1
    return count

total_groups = 0
print(f"{'File':20s}  Groups")
print('-' * 30)
for fname in sorted(os.listdir(TEXT_DIR)):
    if not fname.endswith('.txt'):
        continue
    cnt = count_groups_in_file(os.path.join(TEXT_DIR, fname))
    total_groups += cnt
    print(f"  {fname:18s}  {cnt:4d}")

print(f"\n  Total compound groups: {total_groups}")

---
## 9. Running the Script

Configure the `USER SETTINGS` block in `scripts/compound_lemma_processor.py`, then:

```bash
python3 scripts/compound_lemma_processor.py
```

Key settings:

| Setting | Default | Effect |
|---|---|---|
| `NP_EXPANSION` | `True` | Detect and expand NP-grouped N-at siblings before compound detection |
| `DICT_SEARCH` | `True` | Look up compound form in existing dictionary entries |
| `DICT_REFINE` | `True` | Add missing `.COMPOUND` lines to an existing entry |
| `DICT_ENTRY_CREATE` | `True` | Create a new entry when the compound is absent from the dictionary |
| `OVERWRITE_SOURCE` | `False` | Write `*_processed.txt` to `OUTPUT_FOLDER` instead of editing source |
| `LEMMA_START` | `50001` | Minimum numeric value for new compound IDs |
| `LEMMA_PREFIX` | `"L"` | Prefix for newly generated compound IDs |

The script iterates compound detection up to 20 times per file to handle deeply
nested compounds (e.g. a 4-component noun whose intermediate 2-component results
are themselves eligible for further grouping).